In [1]:
from build123d import *

In [2]:
import filepath_p2

In [3]:
from typing import Dict
from panda3d.core  import (
    NodePath,
    Vec3
)
from panda3d.bullet import (
    BulletRigidBodyNode,BulletConvexHullShape
)
from panda3d.bullet import BulletDebugNode
from panda3d.core import *
from build123d import *
from panda3d.core import NodePath
from util.geometry import *
import torch
import importlib
from cad_tools import cadmesh
importlib.reload(cadmesh)

<module 'cad_tools.cadmesh' from '/mnt/D/Games/GameNotesPanda3D/p4_external/cad_tools/cadmesh.py'>

In [4]:
def trsf_to_matrix(trsf):
    return [
        [trsf.Value(i, j) for j in range(1, 5)]
        for i in range(1, 4)
    ] + [[0,0,0,1]]
from panda3d.core import Mat4

def trsf_to_mat4(trsf, transpose=True):
    """
    Convert OCP gp_Trsf to Panda3D Mat4
    Mat 4 is in transpose to trsf
    """
    m = Mat4(
        trsf.Value(1,1), trsf.Value(1,2), trsf.Value(1,3), trsf.Value(1,4),
        trsf.Value(2,1), trsf.Value(2,2), trsf.Value(2,3), trsf.Value(2,4),
        trsf.Value(3,1), trsf.Value(3,2), trsf.Value(3,3), trsf.Value(3,4),
        0.0,              0.0,              0.0,              1.0
    )

    if transpose:
        m.transposeInPlace()
    return m

In [5]:
class GameObjectFromCAD:
    def __init__(self, build123d_obj):
        self.cad_obj = build123d_obj # as a single obj 
        

    def get_geom(self):
        pass

In [6]:
class BearingDisk(Compound):
    def __init__(
        self,
        diameter: float,
        thickness: float,
        bearing_diameter: float,
        bearing_depth: float,
        shaft_diameter: float,
        parent=None
    ):
        with BuildPart() as disk_part:
            # 主体圆盘
            with BuildSketch():
                Circle(diameter / 2)
            extrude(amount=thickness)

            # 轴承座（浅孔）
            with BuildSketch(Plane.XY):
                Circle(bearing_diameter / 2)
            extrude(amount=-bearing_depth, mode=Mode.SUBTRACT)

            # 中轴孔（贯穿）
            with BuildSketch(Plane.XY):
                Circle(shaft_diameter / 2)
            extrude(amount=thickness, mode=Mode.SUBTRACT)

            # ⚡ 父 joint：固定
            RigidJoint(
                label="shaft_joint",
                joint_location=Location((0, 0, thickness / 2),(0,0,90))
            )

        super().__init__(
            disk_part.part.wrapped,
            joints=disk_part.part.joints,
            parent=parent
        )


In [7]:
import torch
from util.bullet_geometry import *
from util.geometry import *
from art.basic import *
def cube_frame_convex_hull(edge_length,edge_width):
    trape_edge = torch.Tensor([
        [-edge_length / 2, edge_length / 2, -edge_length / 2],
        [edge_length / 2, edge_length / 2, -edge_length / 2],
        [edge_length / 2 - edge_width, edge_length / 2 - edge_width, -edge_length / 2],
        [-edge_length / 2 + edge_width, edge_length / 2 - edge_width, -edge_length / 2]
    ])
    recta_edge = torch.Tensor([
        [-edge_length / 2 + edge_width, edge_length / 2 - edge_width, -edge_length / 2],
        [edge_length / 2 - edge_width, edge_length / 2 - edge_width, -edge_length / 2],
        [edge_length / 2 - edge_width, edge_length / 2 - edge_width, -edge_length / 2 + edge_width],
        [-edge_length / 2 + edge_width, edge_length / 2 - edge_width, -edge_length / 2 + edge_width],
    ])
    # first rotation
    transforms_1edge_to_1surf = [
            # xy
            torch.Tensor([
                [0, 1, 0],
                [-1, 0, 0],
                [0, 0, 1]
            ]),
            torch.Tensor([
                [-1, 0, 0],
                [0, -1, 0],
                [0, 0, 1]
            ]),
            torch.Tensor([
                [0, -1, 0],
                [1, 0, 0],
                [0, 0, 1]
            ]),
            torch.Tensor([
                [1, 0, 0],
                [0, 1, 0],
                [0, 0, 1]
            ])
        ]
    transforms_1surf_to_cube = [
            torch.Tensor([
                [1, 0, 0],
                [0, 1, 0],
                [0, 0, 1]
            ]),
            # xz
            torch.Tensor([
                [0, 0, 1],
                [0, 1, 0],
                [-1, 0, 0]
            ]),
            torch.Tensor([
                [-1, 0, 0],
                [0, 1, 0],
                [0, 0, -1]
            ]),
            torch.Tensor([
                [0, 0, -1],
                [0, 1, 0],
                [1, 0, 0]
            ]),
            # yz
            torch.Tensor([
                [1, 0, 0],
                [0, 0, -1],
                [0, 1, 0]
            ]),
            torch.Tensor([
                [1, 0, 0],
                [0, 0, 1],
                [0, -1, 0]
            ])
        ]
    convex_shapes = []
    trape_surf = batch_transform(
        [trape_edge], transforms_1edge_to_1surf)
    trape_cube = batch_transform(
        trape_surf, transforms_1surf_to_cube)
    recta_surf = batch_transform(
        [recta_edge], transforms_1edge_to_1surf)
    recta_cube = batch_transform(
        recta_surf, transforms_1surf_to_cube)
    # FIXME: use minkowski sum
    for i in range(24):
        # print(i)
        convex_shape = BulletConvexHullShape()
        for point in trape_cube[i]:
            convex_shape.addPoint(Vec3(*point))
        for point in recta_cube[i]:
            convex_shape.addPoint(Vec3(*point))
        convex_shapes.append(convex_shape)
    return convex_shapes
    


In [8]:
class Gimbal(Compound):
    def __init__(
        self, 
        disc_diameter_gap: float,
        disc_thickness_gap: float,
        frame_thickness_diameter: float, 
        corner_radius: float,
        connector_length: float,
        connector_radius: float,
        # disk
    ):
        """
        创建一个用于支撑转盘的 Gimbal 框架
        disc_diameter_gap: 转盘直径 + 留出间隙
        disc_thickness_gap: 转盘厚度 + 留出间隙
        frame_thickness_diameter: 矩形管管径
        corner_radius: 方孔圆角半径
        connector_length: 左右连接柱长度
        connector_radius: 左右连接柱半径
        """
        a = disc_diameter_gap + frame_thickness_diameter/2
        b = disc_thickness_gap + frame_thickness_diameter/2
        r = corner_radius
        tube_radius = connector_radius
        tube_length = connector_length
        # self.disk = disk

        with BuildPart() as gimbal_part:
            # -----------------------------
            # 1️⃣ 构建圆角矩形路径
            with BuildLine() as frame:
                start = (-a/2 + r, -b/2)
                Line(start, (a/2 - r, -b/2))
                JernArc(start=(a/2 - r, -b/2), tangent=(1, 0), radius=r, arc_size=90)
                Line((a/2, -b/2 + r), (a/2, b/2 - r))
                JernArc(start=(a/2, b/2 - r), tangent=(0, 1), radius=r, arc_size=90)
                Line((a/2 - r, b/2), (-a/2 + r, b/2))
                JernArc(start=(-a/2 + r, b/2), tangent=(-1, 0), radius=r, arc_size=90)
                Line((-a/2, b/2 - r), (-a/2, -b/2 + r))
                JernArc(start=(-a/2, -b/2 + r), tangent=(0, -1), radius=r, arc_size=90)
            rectangle_wire = frame.wire()

            # -----------------------------
            # 2️⃣ sweep 圆角矩形管
            # TODO: start and tangent
            start = rectangle_wire @ 0
            tangent = rectangle_wire % 0 
            with BuildSketch(Plane(origin=start, z_dir=tangent)) as sk:
                Circle(frame_thickness_diameter/2)
            sweep(sk.sketch, rectangle_wire, is_frenet=True)

            # -----------------------------
            # 3️⃣ 左右中点圆柱（extrude）沿 X 方向
            left_mid = (-a/2, 0)
            right_mid = (a/2, 0)
        
            for mid, direction in [(left_mid, -1), (right_mid, 1)]:
                plane = Plane(origin=(mid[0], mid[1], 0), z_dir=(direction, 0, 0))
                with BuildSketch(plane) as skc:
                    Circle(connector_radius)
                extrude(skc.sketch, connector_length)
                
            # constraints 
            RevoluteJoint(
                label="bearing_axis",
                axis=Axis((0, 0, 0), (0, 1, 0)),
                angular_range=(0, 360)
            )
            
                                        
        # gimbal_part.part.joints["bearing_axis"].connect_to(disk.joints["shaft_joint"])
        super().__init__(
            gimbal_part.part.wrapped,
            joints=gimbal_part.part.joints,
            # children=[disk]
        )
        


In [9]:
from panda3d.core import Mat4, LVector3f

# 假设平移向量 (x, y, z)
translation = LVector3f(5, -2, 3)

# 创建平移矩阵
m = Mat4.translateMat(translation)
m

LMatrix4f(1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 5, -2, 3, 1)

In [10]:
import numpy as np

In [11]:
from cad_tools.cadmesh import *

In [12]:
from build123d import *
import numpy as np

class Frame(Compound):
    def __init__(self, cube_size, beam,label=None):
        box = Box(cube_size, cube_size, cube_size)
        center = Vector(0, 0, 0)
        joints = []
        with BuildPart() as bp:
            # ========= 1️⃣ 生成框架 =========
            for edge in box.edges():
                mid = edge.position_at(0.5)
                direction = edge.tangent_at(0)
                outward = (mid - center).normalized()
                plane = Plane(
                    origin=mid - outward * (beam / np.sqrt(2)),
                    z_dir=direction
                )
                with BuildSketch(plane):
                    Rectangle(beam, beam)
                sweep(path=edge)

        
        # ========= 2️⃣ 添加 joints =========
        # 六个面
        # face_dirs = [
        #     Vector(1,0,0), Vector(-1,0,0),
        #     Vector(0,1,0), Vector(0,-1,0),
        #     Vector(0,0,1), Vector(0,0,-1),
        # ]
        
        # for i, normal in enumerate(face_dirs):
        #     face_center = normal * (cube_size / 2)
            
        #     joint = RevoluteJoint(
        #         label=f"face_{i}",
        #         parent=bp.part,
        #         location=Location(
        #             Plane(
        #                 origin=face_center,
        #                 z_dir=normal  # 👉 旋转轴方向
        #             )
        #         )
        #     )
            
        #     joints.append(joint)
            x_faces = [
                (Vector(cube_size/2, 0, 0), (0,0,1)),
                (Vector(-cube_size/2, 0, 0), (0,0,-1))
            ]
            for i, (face_center, axis) in enumerate(x_faces):
                RevoluteJoint(
                    label=f"x_face_{i}",
                    axis=Axis(face_center, axis),
                    angular_range=(-90, 90)
                )
        
        # ========= 3️⃣ 初始化 Compound =========
        super().__init__(
            bp.part.wrapped,
            joints=bp.part.joints,
            label=label
        )



In [13]:
sizes = {
    'cube_size' : 100,
    'beam_size' : 5,
    
    'disk_diameter' : 50,
    'disk_thickness': 5,
    'frame_thickness_diameter': 5,
    'gap': 4,
    'corner_radius':2,
    'connector_length':20,
    'connector_radius':2,

    'bearing_diameter': 22.2,
    'bearing_depth': 7,
    'shaft_diameter': 8.2
}
sizes = {
    k:v/10
    for k,v in sizes.items()
}

In [14]:
from panda3d_game.game_object import GameObject,PhysicsGameObject

class FrameObj(PhysicsGameObject):
    def __init__(self,name,mass=1):
        PhysicsGameObject.__init__(self)
        self.name = f"Frame.{name}"
        self.mass = mass
        self.f_cad = Frame(
            sizes['cube_size'],
            sizes['beam_size'],label="frame")
        self.rigid_node = BulletRigidBodyNode(self.name)
        self.rigid_node.setMass(self.mass)
        # TODO: convex hull: 
        convex_shapes = cube_frame_convex_hull(
            edge_length=sizes["cube_size"],
            edge_width=sizes["beam_size"]
        )
        for shape in convex_shapes:
            self.rigid_node.addShape(shape)
        self.rigid_np = NodePath(self.rigid_node)
        
    @property
    def joints(self):
        return self.f_cad.joints

    def render(self,*args, **kwargs):
        mesh_node = build123d_to_panda_node(
            name=self.name,
            part=self.f_cad.solid(), format_=format_default)
        mesh_np = NodePath(mesh_node)
        mesh_np.reparentTo(self.mainPath)

    def compounds(self):
        return self.f_cad

class CMGGimbal(PhysicsGameObject):
    def __init__(self,name):
        PhysicsGameObject.__init__(self)
        self.name = f"cmg_gimbal.{name}"
        self.gimbal_cad = Gimbal(
                disc_diameter_gap=sizes['disk_diameter']+sizes['gap'], 
                disc_thickness_gap=sizes['disk_thickness']+sizes['gap'], 
                frame_thickness_diameter=sizes['frame_thickness_diameter'], 
                corner_radius=sizes['corner_radius'], 
                connector_length=sizes['connector_length'], 
                connector_radius=sizes['connector_radius'],
            )
        self.disk_cad = BearingDisk(
                diameter=sizes['disk_diameter'],
                thickness=sizes['disk_thickness'],
                bearing_diameter=sizes['bearing_diameter'],
                bearing_depth=sizes['bearing_depth'],
                shaft_diameter=sizes['shaft_diameter'],
            )
        self.gimbal_cad \
                .joints["bearing_axis"] \
                .connect_to(self.disk_cad.joints["shaft_joint"])
        self.compound_cad = Compound(
                label=f"gc_{name}", children=[self.gimbal_cad,self.disk_cad], 
            ) 
        self.compound_cad.joints["gimbal_rotation"] = RigidJoint(
                label="gimbal_rotation",
                joint_location=Location((0, 0, 0),(0,90,90)),
                to_part=self.compound_cad
            )
        self.rigid_node = BulletRigidBodyNode(self.name) # node for gimbal
        self.rigid_np = NodePath(self.rigid_node)
        self.disk_rigid_node = BulletRigidBodyNode(f"{self.name}.disk")
        self.disk_rigid_np = NodePath(self.disk_rigid_node)
        self.disk_rigid_np.reparentTo(self.rigid_np)

        # get transform of disk relative to gimbal compound in cad
        disk_transform = trsf_to_mat4(
            self.disk_cad.location.wrapped.Transformation()
        )
        # self.disk_rigid_np.setMat(self.rigid_np, disk_transform)
        # TODO: apply transformation to disk so it is in proper position relative to gimbal
        

        # convec_hull_disk = create_convex_hull_shape_tr(
        convec_hull_disk = create_convex_hull_shape(
            geoms = [build123d_to_panda_geom(self.disk_cad.solid(), format_=format_default)], # TODO get geom TODO tol=
            # transforms = [self.disk_rigid_np.getTransform().getMat()]
        )
        # TODO: check pos 
        self.disk_rigid_node.addShape(convec_hull_disk)
        
        # self.rigid_node.setLinearSleepThreshold(0)
        # self.rigid_node.setAngularSleepThreshold(0)
        # self.rigid_node.setFriction(0)
        
        
        

    def render(self,*args,**kwargs):
        mesh_disk = build123d_to_panda_node(
            name=f"{self.name}.disk.mesh", # FIXME names
            part=self.disk_cad.solid(), format_=format_default)
        mesh_gimbal = build123d_to_panda_node(
            name=f"{self.name}.gimbal",
            part=self.gimbal_cad.solid(), format_=format_default)
        mesh_disk_np = NodePath(mesh_disk)
        mesh_gimbal_np = NodePath(mesh_gimbal)
        mesh_disk_np.reparentTo(self.disk_rigid_np)
        mesh_gimbal_np.reparentTo(self.rigid_np)
        
    @property
    def joints(self):
        return self.compound_cad.joints

    def compounds(self):
        return self.compound_cad
        
        

class ScissorPairCMG(PhysicsGameObject):
    def __init__(self,name):
        PhysicsGameObject.__init__(self)
        self.name = f"sc_cmg.{name}"
        self.gimbals = [
            CMGGimbal(f"{self.name}.gimbal.{i}")
            for i in range(2)
        ]
        self.frame = FrameObj(f"{self.name}.frame")
        for i in range(2):
            self.frame.joints[f"x_face_{i}"].connect_to(
                self.gimbals[i].joints["gimbal_rotation"], 
                angle=0)
        # set transforms 
        # self.rigid_np = self.frame.rigid_np
        for i in range(2):
            transform = trsf_to_mat4(
                self.gimbals[i].compound_cad.location.wrapped.Transformation()
            )
            self.gimbals[i].rigid_np.setMat(transform)      
        
        self.children = [
            self.frame, 
            *self.gimbals 
        ]
    

    def reparentTo(self, other): # rendering
        for child in self.children:
            child.reparentTo(other)

    def render(self,*args,**kwargs):
        for child in self.children:
            child.render()
    
    def compounds(self):
        return Compound(
            label="cmg",
            children=[
                self.frame.compounds(),
                self.gimbals[0].compounds(),self.gimbals[1].compounds()])
        

In [15]:
from panda3d.core import NodePath, LVector3f
from panda3d.core import AmbientLight, DirectionalLight
# from panda3d.stl import load_stl
from direct.showbase.ShowBase import ShowBase
from panda3d_game.app import ContextShowBase
# prc = 
# prc = """

# load-display pandagl
# from 

# """
# loadPrcFileData("",prc)

class MyApp(ContextShowBase):
    def __init__(self, debug=True):
        
        ContextShowBase.__init__(self)
        # 加载 STL 文件
        # stl_node = load_stl("example.stl")  # 返回 GeomNode
        # self.model = NodePath(stl_node)
        # self.model = mesh_np
        # self.model.reparent_to(self.render)
        s = ScissorPairCMG("s")
        s.reparentTo(self.render)
        s.render()

        # 缩放 / 平移 / 旋转
        # self.model.set_scale(0.01)  # 根据 STL 单位调整
        # self.model.set_pos(0, 10, 0)
        # self.model.set_hpr(0, 0, 0)

        # 添加光源
        ambient = AmbientLight("ambient")
        ambient.set_color((0.5, 0.5, 0.5, 1))
        ambient_np = self.render.attach_new_node(ambient)
        self.render.set_light(ambient_np)

        directional = DirectionalLight("dir")
        directional.set_color((1, 1, 1, 1))
        directional_np = self.render.attach_new_node(directional)
        directional_np.set_hpr(-30, -60, 0)
        self.render.set_light(directional_np)

        # 设置背景色
        self.set_background_color(0.1, 0.1, 0.1)
        if debug:
            debug_node = BulletDebugNode('Debug')
            debug_node.showWireframe(True)   # 显示线框
            debug_node.showConstraints(True) # 显示约束
            debug_node.showBoundingBoxes(True)
            debug_node.showNormals(False)
            
            debug_np = NodePath(debug_node)
            debug_np.reparentTo(render)
            debug_np.show()


In [16]:
from qpanda3d import  QPanda3DWidget,  Synchronizer, QControlMultiView
from demos.physics_room import PhyscRoomConsole
class ShaderControl(MyApp, QControlMultiView):
    def __init__(self):
        QControlMultiView.__init__(self)
        MyApp.__init__(self)

    def pause_switch(self):
        pass

from demos.physics_room import PhyscRoomConsole
from ui.qtui import *
from demos.physics_room import PhyscRoomConsole




class ShaderGame(MultiViewQtGUI):
    def get_game(self):
        game = ShaderControl()
        # self.cameras["test"] = game.new_cam_np
        return game

    def get_console(self):
        return PhyscRoomConsole(showbase=self.panda3d)

if __name__ == '__main__':
    # torch.set_printoptions(precision=16, sci_mode=False)
    import sys
    # app = QApplication(sys.argv)
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    window = ShaderGame()
    window.show()
    sys.exit(app.exec_())

:display: loading display module: libpandagl.so
Known pipe types:
  glxGraphicsPipe
(all display modules loaded.)
:ShowBase: Default graphics pipe is glxGraphicsPipe (OpenGL).
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
:display: Created output of type glxGraphicsBuffer
:ShowBase: Successfully opened window of type glxGraphicsBuffer (OpenGL)
:audio: NullAudioManager
:audio: NullAudioManager
/mnt/D/Games/GameNotesPanda3D/p1/py_src/util/maths/differential.py:103: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:63.)
  face_normal = torch.cross(e1, e2)


0 0 1 0
0.965926 0.258819 0 0
-0.258819 0.965926 0 0
5 0 0 1

0 0 -1 0
-0.965926 0.258819 0 0
0.258819 0.965926 0 0
-5 0 0 1



:display: Created output of type GLGraphicsBuffer


{'default': render/camera/cam}


SystemExit: 0

/mnt/D/packages/miniconda3/envs/game-qt6-py312/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
[1,*[1,2]]

In [ ]:
help(create_convex_hull_shape_tr)

In [17]:
s = ScissorPairCMG("s")

0 0.965926 -0.258819 0
0 0.258819 0.965926 0
1 0 0 0
5 0 0 1

0 -0.965926 0.258819 0
0 0.258819 0.965926 0
-1 0 0 0
-5 0 0 1



In [18]:
s.compounds()

Compound at 0x7f20e811f3e0, label(cmg), #children(3)

In [ ]:
s.gimbals[0].compound_cad.

In [19]:
l = s.gimbals[1].compound_cad.location.wrapped.Transformation()

In [21]:
m = trsf_to_matrix(l)

In [22]:
arr = np.array(m)

In [24]:
np.set_printoptions(precision=3, suppress=True)
arr

array([[-0.   , -0.966,  0.259, -5.   ],
       [-0.   ,  0.259,  0.966,  0.   ],
       [-1.   ,  0.   , -0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  1.   ]])

In [28]:
angle_deg = 15
c = np.cos(np.deg2rad(angle_deg))
s = np.sin(np.deg2rad(angle_deg))
c,s

(np.float64(0.9659258262890683), np.float64(0.25881904510252074))

In [20]:
from build123d import Location

In [21]:
lw = l.wrapped

AttributeError: 'OCP.gp.gp_Trsf' object has no attribute 'wrapped'

In [22]:

matrix = [
    [trsf.Value(i, j) for j in range(1, 5)] 
    for i in range(1, 4)
]
# Add the implicit bottom row for a full 4x4 homogeneous matrix
matrix.append([0, 0, 0, 1])

NameError: name 'trsf' is not defined

In [23]:
matrix

NameError: name 'matrix' is not defined

In [24]:
lw

NameError: name 'lw' is not defined

In [25]:
trsf = l.wrapped.Transformation()

AttributeError: 'OCP.gp.gp_Trsf' object has no attribute 'wrapped'

<module 'cad_tools.cadmesh' from '/mnt/D/Games/GameNotesPanda3D/p4_external/cad_tools/cadmesh.py'>

In [18]:
create_convex_hull_shape_tr

<function util.bullet_geometry.create_convex_hull_shape_tr(geoms, transforms)>

In [17]:
cube_size = 100
beam_size = 5
g1 = Gimbal(
    disc_diameter_gap=50+4, 
    disc_thickness_gap=5+4, 
    frame_thickness_diameter=5, 
    corner_radius=2, 
    connector_length=20, 
    connector_radius=2,
)
d1 = BearingDisk(
    diameter=50,
    thickness=5,
    bearing_diameter=22.2,
    bearing_depth=7,
    shaft_diameter=8.2,
)
g2 = Gimbal(
    disc_diameter_gap=50+4, 
    disc_thickness_gap=5+4, 
    frame_thickness_diameter=5, 
    corner_radius=2, 
    connector_length=20, 
    connector_radius=2,
)
d2 = BearingDisk(
    diameter=50,
    thickness=5,
    bearing_diameter=22.2,
    bearing_depth=7,
    shaft_diameter=8.2,
    # parent=g2
)
g1.joints["bearing_axis"].connect_to(d1.joints["shaft_joint"], angle=50)
g2.joints["bearing_axis"].connect_to(d2.joints["shaft_joint"], angle=50)
gc1 = Compound(
    label="gc1", children=[g1,d1], 
)
gc2 = Compound(
    label="gc2", children=[g2,d2], 
)
gc1.joints["gimbal_rotation"] = RigidJoint(
    label="gimbal_rotation",
    joint_location=Location((0, 0, 0),(0,90,90)),
    to_part=gc1
)
gc2.joints["gimbal_rotation"] = RigidJoint(
    label="gimbal_rotation",
    joint_location=Location((0, 0, 0),(0,90,90)),
    to_part=gc2
)
f = Frame(cube_size,beam_size,label="frame")
f.joints["x_face_0"].connect_to(gc1.joints["gimbal_rotation"], angle=90)
f.joints["x_face_1"].connect_to(gc2.joints["gimbal_rotation"], angle=0)

In [110]:
cmg = Compound(label="cmg",children=[f,gc1,gc2])
cmg

Compound at 0x7faabc20a000, label(cmg), #children(3)